# Лабораторная работа №5
## Выполнил студент группы Гавриков Владимир Дмитриевич БПИ2303

### Оглавление
1. [Задание 1](#Задание-№1)
2. [Задание 2](#Задание-№2)
4. [Вывод](#Вывод)

> Дополнительные модули, использованные при выполнение лабораторной

In [9]:
# Импорт необходимых модулей
from datetime import datetime as dt
import re
from collections import defaultdict

### Задание №1
Реализовать методы поиска подстроки в строке. Добавить возможность ввода строки и подстроки с клавиатуры. Предусмотреть возможность существования пробела. Реализовать возможность выбора опции чувствительности или нечувствительности к регистру. Оценить время работы каждого алгоритма поиска и сравнить его со временем работы стандартной функции поиска, используемой в выбранном языке программирования.

#### Алгоритм Кнута-Морриса-Пратта

In [ ]:
# таблица lps - самый длинный префикс, который совпадает с суффиксом, Эта таблица помогает не бегать по тем местам, где уже было совпадение.
def build_lps(pattern):
    lps = [0] * len(pattern)
    length = 0  # длина предыдущего наибольшего префиксного суффикса
    i = 1
    while i < len(pattern):
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        else:
            if length != 0:
                length = lps[length - 1]
            else:
                lps[i] = 0
                i += 1
    return lps
# пример:
# Индекс:     0 1 2 3 4 5 6    lps[2] = 1 → у "aba" совпадает "a" (начало и конец)
# Символ:     a b a b a c a    lps[4] = 3 → у "ababa" совпадает "aba" (начало и конец)
# lps:        0 0 1 2 3 0 1

# имея lps, мы не "откатываемся" на начало при несовпадении. Мы знаем, куда именно прыгнуть.
def kmp_search(text, pattern):
    if not pattern:
        return 0
    lps = build_lps(pattern)
    i = 0  # индекс в text
    j = 0  # индекс в pattern
    while i < len(text):
        if text[i] == pattern[j]:
            i += 1
            j += 1
            if j == len(pattern):
                return i - j  # найдено вхождение
        else:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1
    return -1


#### Упрощенный алгоритм Бойера-Мура

In [ ]:
# обычные алгоритмы работают слева направо, а он начинает с конца подстроки и если  видит несовпадение — сразу перескакивает на много символов вперёд, а не по одному
# пример:
# text:    abcdabcxabcd             начнёт проверку с конца c в abc.
# pattern: abc                      совпадает ли text[i + 2] == pattern[2]
#                                   если нет — он смотрит, где в pattern вообще есть этот "плохой символ". Если нет — перескакивает целиком.

def boyer_moore_search(text, pattern):
    m = len(pattern)
    n = len(text)
    if m == 0:
        return 0

    # Построение таблицы смещений для плохого символа
    bad_char = {}
    for i in range(m - 1):
        bad_char[pattern[i]] = m - i - 1

    i = 0
    while i <= n - m:
        j = m - 1
        # Сравнение символов с конца подстроки
        while j >= 0 and pattern[j] == text[i + j]:
            j -= 1
        if j < 0:
            return i  # найдено вхождение
        else:
            # Смещение по правилу плохого символа:
            shift = bad_char.get(text[i + j], m)
            i += shift
    return -1


In [27]:
# Ввод исходных данных
text = input("Введите строку: ")
pattern = input("Введите подстроку для поиска: ")
case_choice = input("Учитывать регистр? (y/n): ").strip().lower()

if case_choice == 'n':
    text_proc = text.lower()
    pattern_proc = pattern.lower()
else:
    text_proc = text
    pattern_proc = pattern

# Поиск стандартным методом (str.find)
start = dt.now()
result_builtin = text_proc.find(pattern_proc)
time_builtin = (dt.now() - start).total_seconds()

# Поиск алгоритмом Кнута–Морриса–Пратта (КМП)
start = dt.now()
result_kmp = kmp_search(text_proc, pattern_proc)
time_kmp = (dt.now() - start).total_seconds()

# Поиск упрощённым алгоритмом Бойера–Мура
start = dt.now()
result_bm = boyer_moore_search(text_proc, pattern_proc)
time_bm = (dt.now() - start).total_seconds()

# Вывод результатов
print("\nРезультаты поиска подстроки:")
print(f"Стандартный метод: индекс = {result_builtin}, время = {time_builtin:.6f} сек.")
print(f"Алгоритм Кнута–Морриса–Пратта: индекс = {result_kmp}, время = {time_kmp:.6f} сек.")
print(f"Упрощённый алгоритм Бойера–Мура: индекс = {result_bm}, время = {time_bm:.6f} сек.")



Результаты поиска подстроки:
Стандартный метод: индекс = 56, время = 0.001014 сек.
Алгоритм Кнута–Морриса–Пратта: индекс = 56, время = 0.000000 сек.
Упрощённый алгоритм Бойера–Мура: индекс = 56, время = 0.000000 сек.


### Задание №2
Написать программу, определяющую, является ли данное
расположение «решаемым», то есть можно ли из него за конечное число
шагов перейти к правильному. Если это возможно, то необходимо найти хотя
бы одно решение - последовательность движений, после которой числа будут
расположены в правильном порядке.
#### Входные данные: массив чисел, представляющий собой расстановку в
Порядке «слева направо, сверху вниз». Число 0 обозначает пустое поле.
Например, массив [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0] представляет
собой «решенную» позицию элементов.
#### Выходные данные: если решения нет, то функция должна вернуть
Пустой массив []. Если решение есть, то необходимо представить решение —
для каждого шага записывается номер передвигаемого на данном шаге
элемента. 

In [15]:
from collections import deque

def is_solvable(puzzle):
    inv_count = sum(
        1
        for i in range(len(puzzle))
        for j in range(i + 1, len(puzzle))
        if puzzle[i] and puzzle[j] and puzzle[i] > puzzle[j]
    )
    return inv_count % 2 == 0

def get_neighbors(state):
    neighbors = []
    zero_index = state.index(0)
    row, col = divmod(zero_index, 4)
    moves = {
        "up": (row - 1, col),
        "down": (row + 1, col),
        "left": (row, col - 1),
        "right": (row, col + 1),
    }
    for move, (r, c) in moves.items():
        if 0 <= r < 4 and 0 <= c < 4:
            new_state = state[:]
            swap_idx = r * 4 + c
            new_state[zero_index], new_state[swap_idx] = new_state[swap_idx], new_state[zero_index]
            neighbors.append((new_state, new_state[zero_index]))
    return neighbors

def solve_puzzle(start_state):
    goal_state = list(range(1, 16)) + [0]
    if not is_solvable(start_state):
        return []

    queue = deque([(start_state, [])])
    visited = set()
    visited.add(tuple(start_state))

    while queue:
        state, path = queue.popleft()
        if state == goal_state:
            return path

        for new_state, moved_tile in get_neighbors(state):
            state_tuple = tuple(new_state)
            if state_tuple not in visited:
                visited.add(state_tuple)
                queue.append((new_state, path + [moved_tile]))

    return []

puzzle = list(map(int, input("Введите 16 чисел через пробел: ").split()))
solution = solve_puzzle(puzzle)
if solution:
    print("Решение найдено:", solution)
else:
    print("Решение отсутствует.")


Решение найдено: [7, 11, 15]


### Вывод

В обоих заданиях мы использовали алгоритмы, которые позволяют не просто "решать", а делать это оптимально и эффективно, опираясь на структуру данных и заранее рассчитанные подсказки.